In [ ]:
# import fiscalyear
# fiscalyear.setup_fiscal_calendar(start_month=10)
# from fiscalyear import *
# FiscalYear.current()
# FiscalYear.current().start
# FiscalYear.current().end
# fydf = s._df[['dateTime', 'month-day', 'value']]
# fydf.iloc[80:, :].head(15)
# fydf.apply
# fydf['dateTime'] = pd.to_datetime(fydf['dateTime'])
# fydf['FiscalDate'] = fydf['dateTime'].apply(lambda x: FiscalDateTime(x.year, x.month, x.day))
# fydf['FiscalDate'][0].year
# fydf['FiscalYear'] = fydf['FiscalDate'].apply(lambda x: x.year)
# fydf
# fy2023 = fydf[fydf['FiscalDate'].apply(lambda x: x.year) == 2023]
# fy2023

# fig, ax = plt.subplots(figsize=(9, 7))

# for year in fydf['FiscalYear'].unique():
# #             ax.plot(data['month-day'], data['value'])
# #             ax.set_xticks(forced_x_positions[0])
# #             ax.set_xticklabels(forced_x_labels, rotation=45)
#             grouped_water_years = fydf.groupby('FiscalYear')
#             df = grouped_water_years.get_group(year)
#             ax.plot(df['month-day'], df['value'], label=f'{year}', alpha=.5)

# plt.show()            

In [ ]:
# 

In [ ]:
# def convert_to_fiscal_year(date):
#     fy = date.year if date.month >= 10 else date.year -1
    
#     #Adjust day if out of range for the month and account for leap year
#     max_day = pd.Timestamp(year=date.year, month=date.month, day=1).days_in_month
#     if date.month ==2 and pd.Timestamp(date).is_leap_year:
#         max_day=29
    
#     day = min(date.day, max_day)
#     print(day)
#     return FiscalDate(fy, date.month, day)

# fydf['fiscal_date'] = fydf['dateTime'].apply(convert_to_fiscal_year)

In [1]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pandas as pd
import math
import datetime
from webservices import nrcs_snotel, usgs_streamflow

In [2]:
class DataLoader:
    def __init__(self, cvs_path_or_df, name_of_date_column, name_of_Q_column):
        self.cvs_path_or_df = cvs_path_or_df
        self._name_of_date_column = name_of_date_column
        self._name_of_Q_column = name_of_Q_column
        self.df = self.load_data()
    
    def load_data(self):
        if isinstance(self.cvs_path_or_df, pd.DataFrame):
            self._df = self.cvs_path_or_df.copy()  # Set the DataFrame directly
            # Convert columns to numeric, handling errors by setting non-numeric values to NaN
            if isinstance(self._df[self._name_of_Q_column][0], str):
                self._df[self._name_of_Q_column] = pd.to_numeric(self._df[self._name_of_Q_column], errors='coerce')

        elif isinstance(cvs_path_or_df, str):
            try:
                print(f"Importing CSV with the file path: {csv_path}")
                self._df = pd.read_csv(cvs_path_or_df)
                # Convert columns to numeric, handling errors by setting non-numeric values to NaN
                if isinstance(self._df[self._name_of_Q_column][0], str):
                    self._df[self._name_of_Q_column] = pd.to_numeric(self._df[self._name_of_Q_column], errors='coerce')
                # self._df['Year'] = pd.to_numeric(self._df['Year'], errors='coerce')
            except Exception as e:
                print(f"Error reading CSV file: {e}")
                
        else:
            print("Invalid input type. Please provide either a pandas DataFrame or a CSV file path.")
            


In [3]:
class StatisticsCalculator:
    def __init__(self, DataLoader):
        self.data_loader = DataLoader
        self._df = self.data_loader._df
        self._df['Date'] = pd.to_datetime(self._df[self.data_loader._name_of_date_column])
        self._df = self._df[~self._df.duplicated('Date')]
        #Replace year in regular calendar year with water year.  This is the key line of code/method for plotting WY.
        self._df['WY_Date'] = self._df['Date'].apply(lambda x: x.replace(year=x.year+1) if 10 <= x.month <=12 else x)
        self._df['month'] = self._df['WY_Date'].dt.month
        self._df['Water Year'] = self._df['WY_Date'].dt.year
        self._df['Calendar Year'] = self._df['Date'].dt.year
        self._df['month-day'] = self._df['WY_Date'].apply(lambda x: x.strftime('%m-%d'))
#         self._df['Water Year'] = self._df['Date'].dt.year.where(self._df['Date'].dt.month<10, self._df['Date'].dt.year+1)
        self.stats = self._calculate_statistics()
#         if kwargs.get('water_year_on', True):
        self._grouped_water_years = self._df.groupby('Water Year')
        
#         else:
#         self._grouped_calendar_years = self._df.groupby('Calendar Year')
        #Create new column with water year dates by replacing normal calendar year in dt object with water year 
        
    def _calculate_statistics(self, **kwargs):

        self._stats = self._df.groupby("month-day")[self.data_loader._name_of_Q_column].agg(['mean', 'median', 'std', ("q25", lambda x: x.quantile(0.25)), ("q75", lambda y: y.quantile(0.75))])
        self._stats.reset_index(inplace=True)
        self._stats['month'] = self._stats['month-day'].str[:2]
#         self._stats['month'] = pd.to_datetime(self._stats['month-day'], format='%m')
        self._stats['month'] = self._stats['month'].fillna('')
        water_year_sort_order = ['10', '11', '12', '01', '02', '03', '04', '05', '06', '07', '08', '09']
        self._stats['water_year_sort'] = self._stats['month'].map({month: i for i, month in enumerate(water_year_sort_order)})
        self._stats = self._stats.sort_values(by='water_year_sort')
        self._stats.reset_index(inplace=True)
#         self._monthly_stats = self._df.groupby("month")[self.data_loader._name_of_Q_column].agg(['mean', 'median', 'std', ("q25", lambda x: x.quantile(0.25)), ("q75", lambda y: y.quantile(0.75))])
        self._mean = self._stats.loc[:, 'mean']
        self._median = self._stats.loc[:, 'median']
        self._st_dev = self._stats.loc[:, 'std']
        self._percentile25 = self._stats.loc[:, 'q25']
        self._percentile75 = self._stats.loc[:, 'q75']
        self._lower_bound_st_dev = self._mean - self._st_dev
        self._upper_bound_st_dev = self._mean + self._st_dev
        self._lower_bound_percentile25 = self._mean - self._percentile25
        self._upper_bound_percentile75 = self._mean + self._percentile75
        



    

In [38]:
import scipy.integrate as integrate
# from scipy.stats import kendalltau
from statsmodels.tsa.seasonal import STL
# from pymannkendall import seasonal_test
from pymannkendall import original_test

class StreamflowClimateStatistics(StatisticsCalculator):
    def __init__(self, DataLoader):
        super().__init__(DataLoader)
        
        yearly_volume_dfs = {}
        yearly_volume_statistics = {}
        
        for wy in self._df['Water Year'].unique():           
            volume_df = pd.DataFrame()
            volume_df['values'] = self._df[self._df['Water Year']==wy][self.data_loader._name_of_Q_column]
            volume_df['WY_Date'] = self._df[self._df['Water Year']==wy]['WY_Date']
            volume_df['daily_volume_ft^3/d'] = volume_df['values'] * 86400 / 43559.9 / 1000000 #convert ft^3 to maf.
            volume_df['cumsum_volume ft^3'] = volume_df['daily_volume_ft^3/d'].cumsum()
#             print(volume_df)                  
            total_volume = volume_df['daily_volume_ft^3/d'].sum()
            wy_dates = self._df[self._df['Water Year']==wy]['WY_Date']
            half_volume_point_date = wy_dates[volume_df['cumsum_volume ft^3']>= 0.5 * total_volume].iloc[0]
#             tau, p_value = kendalltau(volume_df['WY_Date'], volume_df['values'])
#             df_for_mk_test = volume_df[['WY_Date']] #, 'values']]
#             df_for_mk_test['WY_Date'].astype(int) #need convert date to integer for test to work.
#             smk = seasonal_test(df_for_mk_test, alpha=0.05, period=12)
            yearly_volume_dfs[wy] = volume_df
            yearly_volume_statistics[wy] = {'total_volume': total_volume, 
                                            'half_volume': total_volume * 0.5, 
                                            'half_volume_point_date': half_volume_point_date,
#                                             'mann-kendall_tau': tau,
#                                             'mann-kendall_p-value': p_value,
#                                             'seasonal-mann-kendall': {
#                                                 'water_year': wy,
#                                                 'trend': smk.trend,
#                                                 'h': smk.h, #True (if trend is present) or False (if the trend is absence)
#                                                 'p_value': smk.p,
#                                                 'z': smk.z, #normalized test stats
#                                                 'tau': smk.Tau,
#                                                 'Mann-Kendall score': smk.s
#                                              }
                                            }

        self.yearly_volume_statistics = pd.DataFrame(yearly_volume_statistics).T
        self.yearly_volume_statistics['half_volume_point_date'] = self.yearly_volume_statistics['half_volume_point_date']
        self.yearly_volume_statistics['half_volume_point_day_of_yr'] = self.yearly_volume_statistics['half_volume_point_date'].apply(lambda x: x.strftime('%m-%d'))     
        self.yearly_volume_dfs = yearly_volume_dfs
#         def _to_day_of_year(dt):
#             return dt.timetuple().tm_yday
        self.half_volume_day_mann_kendall_test = original_test(self.yearly_volume_statistics['half_volume_point_date'].apply(lambda dt: dt.timetuple().tm_yday), alpha=0.05)
        self.total_volume_mann_kendall_test = original_test(self.yearly_volume_statistics['total_volume'])

    def _STL(self):
        self.stl = STL(self.volume_df[['WY_Date', 'values']], seasonal=13)
        self.stl_result = stl.fit()
        self.trend = result.trend
        
            

In [46]:
# class PlotStreamflowClimateStatistics():
# #     def __init__(self, StreamflowClimateStatistics):
# #         self.vol_stats = StreamflowClimateStatistics.yearly_volume_statistics
        
#     def plot_half_volume_days(self, site_name):
#         def _to_day_of_year(dt):
#             return dt.timetuple().tm_yday
#         x = self.vol_stats.index
# #             x = volume_stats_for_vip_sites[12452500].index
# #             volume_stats_for_vip_sites[12452500]['half_volume_point_day_of_yr'] = volume_stats_for_vip_sites[12452500]['half_volume_point_date'].apply(_to_day_of_year)
#         self.vol_stats['half_volume_point_day_of_yr'] = self.vol_stats['half_volume_point_date'].apply(lambda dt: dt.timetuple().tm_yday)
#         y = self.vol_stats['half_volume_point_day_of_yr']

#         fig = go.Figure()

#         labels = self.vol_stats['half_volume']
#         fig.add_trace(go.Scatter(x=x, y=y,mode='markers'))#, name=self.vol_stats['half_volume_point_date']))
#         fig.update_layout(title=site_name)
#         self.half_vol_days_fig = fig
#         fig.show()
    
#     def plot_total_yearly_volumes(self):
#         fig = go.Figure()
#         fig.add_trace(go.Scatter(x=self.vol_stats.index, y=self.vol_stats['total_volume']))
#         self.total_yearly_vol_fig = fig
#         fig.show()

In [47]:
class Plot_HalfVolumeDays():
    def __init__(self, StreamflowClimateStatistics):
        self.vol_stats = StreamflowClimateStatistics.yearly_volume_statistics

    def plot_half_volume_days(self, site_name):
        def _to_day_of_year(dt):
            return dt.timetuple().tm_yday
        x = self.vol_stats.index
#             x = volume_stats_for_vip_sites[12452500].index
#             volume_stats_for_vip_sites[12452500]['half_volume_point_day_of_yr'] = volume_stats_for_vip_sites[12452500]['half_volume_point_date'].apply(_to_day_of_year)
        self.vol_stats['half_volume_point_day_of_yr'] = self.vol_stats['half_volume_point_date'].apply(_to_day_of_year)
        y = self.vol_stats['half_volume_point_day_of_yr']

        fig = go.Figure()

        labels = self.vol_stats['half_volume']
        fig.add_trace(go.Scatter(x=x, y=y,mode='markers'))#, name=self.vol_stats['half_volume_point_date']))
        fig.update_layout(title=site_name)
        fig.show()
        self.fig = fig
#         return fig

In [48]:
class Plot_TotalYearlyVolumes():
    def __init__(self, StreamflowClimateStatistics):
        self.vol_stats = StreamflowClimateStatistics.yearly_volume_statistics

    def plot_total_yearly_volumes(self):
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=self.vol_stats.index, y=self.vol_stats['total_volume']))
        self.fig = fig
        fig.show()

In [7]:
# class Plot_Trend():
#     def __init__(self, StreamflowClimateStatistics):
#         self.vol_stats = StreamflowClimateStatistics.yearly_volume_statistics
           

In [49]:
vip_sites = {
    #Upper Tribs
    #'Okanogan River Near Tonasket, WA': 12445000,
    'Chelan River at Chelan': 12452500,
    #Yakima
    'Yakima at Kiona': 12510500,
    'Yakima River at Umtanum, WA': 12484500,
    #Pend Orielle
    'Clark Fork below Missoula MT': 12353000,
    'Clark Fork at St. Regis MT': 12354500,
    'Flathead at Columbia Falls MT': 12363000,
    #Spokane/Coeur D Alene
    'Spokane River near Post Falls, ID':  12419000,
    'Coeur D Alene River nr Cataldo, ID': 12413500,
    # Clearwater and Salmon
    'Clearwater River at Orofino ID': 13340000,      
    'MF Salmon River at Mouth Nr Shoup, ID': 13310199,
    #Snake River
        #Upper/headwaters
        #Mid/Boise
        #Lower
    'Snake River at Anatone': 13334300,
#     'Grande Ronde River at Troy, OR': 13333000,  #doesn't have 2024 data and causes script to crash...prob add error handling
#     'Snake River at Hells Canyon Dam': 13290450,
    'Snake River at Weiser ID': 13269000, 
    #Interior OR Basins
    'Owyhee River nr Rome OR': 13181000,
    'John Day at McDonald Ferry': 14048000,
    'John Day at Service Creek': 14046500,
    'Deschutes at Moody': 14103000,
    'Umatilla nr Pendleton': 14020850,
    #Cascades
    'Klickitat': 14113000,
    'Hood River': 14120000,
    'White Salmon': 14123500,
    'Walla Walla nr Touchet': 14018500,
}

In [51]:
volume_stats_for_vip_sites = {}

for site in vip_sites.items():
    print(site)
    print(site[1])
    data = usgs_streamflow().get_data(start_date='1921-10-01', end_date=datetime.datetime.now().strftime('%Y-%m-%d'),sites=str(site[1])).reset_index()
    a = DataLoader(data, 'Date', name_of_Q_column='Discharge')
    s = StreamflowClimateStatistics(a)
#     print(s.yearly_volume_dfs)
#     print(s.yearly_volume_statistics)
#     print(s.yearly_volume_statistics)
    volume_stats_for_vip_sites[site[1]] = s.yearly_volume_statistics
    print(f'Mann-Kendall Test Results looking for trend in 50% runoff timing: {s.half_volume_day_mann_kendall_test}')
    print(f'Mann-Kendall Test Results looking for trend in total annual discharge: {s.total_volume_mann_kendall_test}')
    plot = Plot_HalfVolumeDays(s)
    plot.plot_half_volume_days(site[0])
    plot2 = Plot_TotalYearlyVolumes(s)
    plot2.plot_total_yearly_volumes()
#     with open('plots.html', 'a') as f:
#         f.write(plot.fig.to_html(full_html=False, include_plotlyjs='cdn'))
#         f.write(plot2.fig.to_html(full_html=False, include_plotlyjs='cdn'))

    

('Chelan River at Chelan', 12452500)
12452500
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12452500&startDT=1921-10-01&endDT=2024-05-14&siteStatus=all&parameterCd=00060
Mann-Kendall Test Results looking for trend in 50% runoff timing: Mann_Kendall_Test(trend='decreasing', h=True, p=0.0003328812621694599, z=-3.5882685942368413, Tau=-0.23986293546544832, s=-1260.0, var_s=123106.66666666667, slope=-0.4666666666666667, intercept=134.8)
Mann-Kendall Test Results looking for trend in total annual discharge: Mann_Kendall_Test(trend='no trend', h=False, p=0.15754492042061452, z=1.4133769460840924, Tau=0.09461260232248239, s=497.0, var_s=123153.66666666667, slope=0.0017405236651140176, intercept=1.392814214982128)


('Yakima at Kiona', 12510500)
12510500
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12510500&startDT=1921-10-01&endDT=2024-05-14&siteStatus=all&parameterCd=00060
Mann-Kendall Test Results looking for trend in 50% runoff timing: Mann_Kendall_Test(trend='no trend', h=False, p=0.05617595276951448, z=-1.9096681620227114, Tau=-0.1354515050167224, s=-567.0, var_s=87845.0, slope=-0.21226533166458073, intercept=96.65807259073843)
Mann-Kendall Test Results looking for trend in total annual discharge: Mann_Kendall_Test(trend='no trend', h=False, p=0.5194424930840069, z=-0.6442050246294868, Tau=-0.04586717630195891, s=-192.0, var_s=87906.0, slope=-0.002163376033863222, intercept=2.5545655290355413)


('Yakima River at Umtanum, WA', 12484500)
12484500
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12484500&startDT=1921-10-01&endDT=2024-05-14&siteStatus=all&parameterCd=00060
Mann-Kendall Test Results looking for trend in 50% runoff timing: Mann_Kendall_Test(trend='decreasing', h=True, p=0.03689981273883314, z=-2.0868708254932917, Tau=-0.13953931086997906, s=-733.0, var_s=123035.66666666667, slope=-0.1111111111111111, intercept=147.66666666666666)
Mann-Kendall Test Results looking for trend in total annual discharge: Mann_Kendall_Test(trend='no trend', h=False, p=0.07727493406395092, z=1.7667211826051155, Tau=0.1182181610508281, s=621.0, var_s=123153.66666666667, slope=0.002798157250131425, intercept=1.5303219532184416)


('Clark Fork below Missoula MT', 12353000)
12353000
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12353000&startDT=1921-10-01&endDT=2024-05-14&siteStatus=all&parameterCd=00060
Mann-Kendall Test Results looking for trend in 50% runoff timing: Mann_Kendall_Test(trend='no trend', h=False, p=0.4286115176580563, z=-0.791569875794939, Tau=-0.05531914893617021, s=-247.0, var_s=96581.0, slope=-0.034482758620689655, intercept=142.6206896551724)
Mann-Kendall Test Results looking for trend in total annual discharge: Mann_Kendall_Test(trend='no trend', h=False, p=0.7821657547574354, z=0.2764977927104695, Tau=0.01948488241881299, s=87.0, var_s=96741.66666666667, slope=0.0013183501339534844, intercept=3.802423274617251)


('Clark Fork at St. Regis MT', 12354500)
12354500
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12354500&startDT=1921-10-01&endDT=2024-05-14&siteStatus=all&parameterCd=00060
Mann-Kendall Test Results looking for trend in 50% runoff timing: Mann_Kendall_Test(trend='no trend', h=False, p=0.18248039843483443, z=-1.3331566607805194, Tau=-0.09152114454029034, s=-435.0, var_s=105978.33333333333, slope=-0.05128205128205128, intercept=143.48717948717947)
Mann-Kendall Test Results looking for trend in total annual discharge: Mann_Kendall_Test(trend='no trend', h=False, p=0.8683598406888478, z=-0.16574219216749408, Tau=-0.011571638964864297, s=-55.0, var_s=106150.33333333333, slope=-0.0011438042787058251, intercept=5.201740867173705)


...

('Umatilla nr Pendleton', 14020850)
14020850
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=14020850&startDT=1921-10-01&endDT=2024-05-14&siteStatus=all&parameterCd=00060
Mann-Kendall Test Results looking for trend in 50% runoff timing: Mann_Kendall_Test(trend='no trend', h=False, p=0.10651757220332425, z=1.6140454317846018, Tau=0.21428571428571427, s=87.0, var_s=2839.0, slope=0.75, intercept=74.5)
Mann-Kendall Test Results looking for trend in total annual discharge: Mann_Kendall_Test(trend='no trend', h=False, p=0.6125283803487118, z=-0.506467669601787, Tau=-0.06896551724137931, s=-28.0, var_s=2842.0, slope=-0.001632946436476226, intercept=0.39875368328888844)


('Klickitat', 14113000)
14113000
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=14113000&startDT=1921-10-01&endDT=2024-05-14&siteStatus=all&parameterCd=00060
Mann-Kendall Test Results looking for trend in 50% runoff timing: Mann_Kendall_Test(trend='no trend', h=False, p=0.08542874779252041, z=-1.7200203211435308, Tau=-0.11929824561403508, s=-544.0, var_s=99662.66666666667, slope=-0.09449404761904762, intercept=96.98846726190476)
Mann-Kendall Test Results looking for trend in total annual discharge: Mann_Kendall_Test(trend='no trend', h=False, p=0.9117882963267614, z=-0.11078316402295861, Tau=-0.007894736842105263, s=-36.0, var_s=99813.33333333333, slope=-0.00017145122847726628, intercept=1.1013682612322107)


('Hood River', 14120000)
14120000
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=14120000&startDT=1921-10-01&endDT=2024-05-14&siteStatus=all&parameterCd=00060
Mann-Kendall Test Results looking for trend in 50% runoff timing: Mann_Kendall_Test(trend='no trend', h=False, p=0.6366418477948277, z=0.4723992385142055, Tau=0.0423728813559322, s=75.0, var_s=24538.333333333332, slope=0.07142857142857142, intercept=64.89285714285714)
Mann-Kendall Test Results looking for trend in total annual discharge: Mann_Kendall_Test(trend='no trend', h=False, p=0.33551383376681354, z=-0.9630671342563537, Tau=-0.08587570621468926, s=-152.0, var_s=24583.333333333332, slope=-0.0013081667330611798, intercept=0.7001147513246453)


('White Salmon', 14123500)
14123500
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=14123500&startDT=1921-10-01&endDT=2024-05-14&siteStatus=all&parameterCd=00060
Mann-Kendall Test Results looking for trend in 50% runoff timing: Mann_Kendall_Test(trend='no trend', h=False, p=0.21601724760471974, z=-1.2371881289924578, Tau=-0.08451865594722738, s=-410.0, var_s=109288.66666666667, slope=-0.078125, intercept=90.828125)
Mann-Kendall Test Results looking for trend in total annual discharge: Mann_Kendall_Test(trend='no trend', h=False, p=0.7762764820380559, z=0.2841747273921827, Tau=0.019583591012162442, s=95.0, var_s=109417.0, slope=0.00017867022520384835, intercept=0.7849170636296227)


('Walla Walla nr Touchet', 14018500)
14018500
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=14018500&startDT=1921-10-01&endDT=2024-05-14&siteStatus=all&parameterCd=00060
Mann-Kendall Test Results looking for trend in 50% runoff timing: Mann_Kendall_Test(trend='no trend', h=False, p=0.07238024176360969, z=1.796718995562047, Tau=0.14383561643835616, s=378.0, var_s=44027.333333333336, slope=0.16666666666666666, intercept=55.0)
Mann-Kendall Test Results looking for trend in total annual discharge: Mann_Kendall_Test(trend='no trend', h=False, p=0.8080989995393981, z=0.2428791737133461, Tau=0.0197869101978691, s=52.0, var_s=44092.0, slope=0.0002900085366849519, intercept=0.4037060318582655)


In [44]:
s.total_volume_mann_kendall_test

Mann_Kendall_Test(trend='decreasing', h=True, p=0.027146416748168978, z=-2.209405961553447, Tau=-0.1548844657973004, s=-677.0, var_s=93614.33333333333, slope=-0.14634146341463414, intercept=114.8048780487805)

In [ ]:
data = PGPWData.getUSGSdata(start_date='1900-10-01', end_date=datetime.datetime.now().strftime('%Y-%m-%d'),sites=12452500).reset_index()
a = DataLoader(data, 'dateTime', name_of_Q_column='value')
s = StreamflowClimateStatistics(a)

In [ ]:
plot = Plot_HalfVolumeDays(s)

In [ ]:
plot.vol_stats

In [ ]:
plot.plot_half_volume_days()

In [ ]:
fig, ax = plt.subplots(figsize=(7,9))
x = volume_stats_for_vip_sites[12452500].index
y = volume_stats_for_vip_sites[12452500]['half_volume_point_day_of_yr']
labels = volume_stats_for_vip_sites[12452500]['half_volume']

ax.plot(x, y, 'o')
# for i, txt in enumerate(labels):
#     ax.annotate(txt, (x[i], y[i]))
#     ax.text(x,y, str(labels))
fig.show()

In [ ]:
s.yearly_volume_statistics

In [ ]:
class PlotUtils:
    def __init__(self, StatisticsCalculator, **kwargs):
        self._StatisticsCalculator = StatisticsCalculator
        self._colors = ['crimson', 'springgreen', 'dodgerblue', 'purple', 'green', 'deeppink', "lawngreen", "coral", "lime", "navy", "goldenrod", 'crimson', 'springgreen', 'dodgerblue', 'purple', 'green', 'deeppink', "lawngreen", "coral", "lime", "navy", "goldenrod"]
        self._name_of_date_column = StatisticsCalculator.data_loader._name_of_date_column
        self._name_of_Q_column = StatisticsCalculator.data_loader._name_of_Q_column
        
#fiiiiii
    def _plot_individual_years(self, ax, series_alpha, kwargs):
#         print(grouped_water_years)
        if kwargs.get('water_year_on', True):
            for year in self._StatisticsCalculator._df['Water Year'].unique():
                year_df = self._StatisticsCalculator._grouped_water_years.get_group(year) 
                ax.plot(year_df['month-day'], year_df[self._name_of_Q_column], label=f'{year}', alpha=series_alpha)
        else:
            for year in self._StatisticsCalculator._df['Calendar Year'].unique():
#                 year_df = self._StatisticsCalculator._df[self._StatisticsCalculator._df['Calendar Year']==year]
                year_df = self._StatisticsCalculator._grouped_calendar_years.get_group(year) 
                ax.plot(year_df['month-day'], year_df[self._name_of_Q_column], label=f'{year}', alpha=series_alpha)

    def _plot_central_tendency_stats(self, ax, plot_stats):
        if plot_stats:
            self._StatisticsCalculator._mean.plot(ax=ax, label="Mean", linestyle=':', color='black', linewidth=1, zorder=3)
            self._StatisticsCalculator._median.plot(ax=ax, label="Median", linestyle='--', color='black', linewidth=1, zorder=3)

    def _plot_highlighted_years(self, ax, highlight_years, **kwargs):
        if isinstance(kwargs.get('highlight_years'), list):
            for year in highlight_years:
    #             if kwargs.get('water_year_on', True):
                year_df = self._StatisticsCalculator._grouped_water_years.get_group(year)
    #             else:
    #                 year_df = self._StatisticsCalculator._df[self._StatisticsCalculator._df['Calendar Year']==year]

                ax.plot(year_df['month-day'], year_df[self._name_of_Q_column], label=f'{year}', color='red', linewidth=1.5, alpha=1)
        else:
            pass
    def _plot_spread(self, **kwargs):
        
        if kwargs.get('spread_on', True):            
            spread_zorder = kwargs.get('spread_zorder', 1)
            spread_alpha = kwargs.get('spread_alpha', 0.15)
            spread_color = kwargs.get('spread_color', 'yellow')
            
            if kwargs.get('spread_type', 'quartiles'):
                lower_bound = self._StatisticsCalculator._lower_bound_percentile25
                upper_bound = self._StatisticsCalculator._upper_bound_percentile75
            elif kwargs.get('spread_type', 'std'):
                lower_bound = self._StatisticsCalculator._lower_bound_st_dev
                upper_bound = self._StatisticsCalculator._upper_bound_st_dev

            plt.fill_between(
                list(range(0, len(pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0]))),
                pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0].astype(float),
                pd.DataFrame(lower_bound).iloc[:, 0].astype(float),                
        #                 pd.DataFrame(self._StatisticsCalculator._lower_bound_st_dev).iloc[:, 0].astype(float),
                where=(pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0].astype(float) > pd.DataFrame(self._StatisticsCalculator._lower_bound_percentile25).iloc[:, 0].astype(float)),
                interpolate=True, color=spread_color, alpha=spread_alpha, zorder=spread_zorder, label="q25-q75")

            plt.fill_between(
                list(range(0, len(pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0]))),
                pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0].astype(float),
                pd.DataFrame(upper_bound).iloc[:, 0].astype(float),
        #                 pd.DataFrame(self._StatisticsCalculator._upper_bound_st_dev).iloc[:, 0].astype(float),                
                where=(pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0].astype(float) < pd.DataFrame(self._StatisticsCalculator._upper_bound_percentile75).iloc[:, 0].astype(float)),
                interpolate=True, color=spread_color, alpha=spread_alpha, zorder=spread_zorder)
        else:
            pass
    def _plot_grouped_by_decade(self, ax, kwargs):
        for i, decade in enumerate(self._StatisticsCalculator._unique_decades):
            years_in_decade = [year for year in self._StatisticsCalculator._unique_years if decade <= year < decade + 10]
            mean_values = self._StatisticsCalculator._pivot_table[years_in_decade].mean(axis=1)
            std_dev_values = self._StatisticsCalculator._pivot_table[years_in_decade].std(axis=1)
            ax.plot(self._StatisticsCalculator._pivot_table.index, mean_values, label=f'Decade {decade}s', color=self._colors[i])
            ax.fill_between(self._StatisticsCalculator._pivot_table.index, mean_values - std_dev_values, mean_values + std_dev_values, alpha=0.2, color=self._colors[i])
    

    def _customize_plot(self, ax, kwargs):
        if kwargs.get('water_year_on', True):
            self._forced_x_labels = kwargs.get('forced_x_labels', ['10-01', '11-01', '12-01', '01-01', '02-01', '03-01', '04-01', '05-01', '06-01', '07-01', '08-01', '09-01'])
            self._forced_x_positions = kwargs.get('forced_x_positions', [1, 31, 62, 93, 121, 152, 182, 213, 243, 273, 303, 335]),
        else:
            self._forced_x_labels = kwargs.get('forced_x_labels', ['01-01', '02-01', '03-01', '04-01', '05-01', '06-01', '07-01', '08-01', '09-01', '10-01', '11-01', '12-01'])       
            self._forced_x_positions = kwargs.get('forced_x_positions', [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 336]),

        if self._forced_x_positions is not None and self._forced_x_labels is not None:
#             print(self._forced_x_positions[0])
#             print(self._forced_x_labels)
            ax.set_xticks(self._forced_x_positions[0])
            ax.set_xticklabels(self._forced_x_labels, rotation=45)
            xlim_min = self._forced_x_positions[0][0]
            xlim_max = self._forced_x_positions[0][-1]
            ax.set_xlim([xlim_min, xlim_max])
            ax.set_ylim([kwargs.get('y_lower_lim', self._StatisticsCalculator._df[self._StatisticsCalculator.data_loader._name_of_Q_column].min()), kwargs.get('y_upper_lim', self._StatisticsCalculator._df[self._StatisticsCalculator.data_loader._name_of_Q_column].max())])

        plt.grid(color='green', linestyle=":", linewidth=0.5)
        plt.xlabel('Month-Day')
        plt.ylabel(kwargs.get('ylabel', "Discharge (cfs)"))
        plt.title(kwargs.get('title'))

        if kwargs.get('legend_mode') == "partial":
#             labels = ['Mean', 'Median', 'q25-q75'] + kwargs.get('highlight_years', [])
            plt.legend(loc=kwargs.get('legend_pos', 'upper right'), ncol=kwargs.get('legend_ncol', 1), labels=['Mean', 'Median'] + kwargs.get('highlight_years', []))
        else:
            plt.legend(loc=kwargs.get('legend_pos'), ncol=kwargs.get('legend_ncol'))#, labels=labels)

In [ ]:
class StackedLinePlots:
    '''
    Keyword Arguments and defaults include:
        plot_central_tendency_stats=True,
        highlight_years=[],
        water_year=True,
        quartile_shading=True,
        quartile_shading_alpha=0.5,
        group_by_decade=False,
        series_alpha=0.3,
        quartile_shading_zorder=1,
        'forced_x_positions'=[1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 336],
        'forced_x_labels'=['01-01', '02-01', '03-01', '04-01', '05-01', '06-01', '07-01', '08-01', '09-01', '10-01', '11-01', '12-01']
        'y_lower_lim'=0,
        'y_upper_lim'=25,
        'ylabel'='Discharge',
        'title',
        'legend_mode'='partial',
        'legend_pos'='upper right',
        'legend_ncol'=1
    '''
    def __init__(self, StatisticsCalculator, **kwargs):
        self._StatisticsCalculator = StatisticsCalculator
        self._PlotUtils = PlotUtils(StatisticsCalculator)
            
        # self._prepare_data_for_plotting(kwargs.get('input_start_year', 2010), kwargs.get('input_end_year', 2020))

        fig, ax = plt.subplots(figsize=(9, 7))

        self._PlotUtils._plot_central_tendency_stats(ax, kwargs.get('plot_central_tendency_stats', True))
        self._PlotUtils._plot_highlighted_years(ax, kwargs.get('highlight_years'))
        self._PlotUtils._plot_spread(**kwargs)

        if kwargs.get('group_by_decade', False):
            self._PlotUtils._plot_grouped_by_decade(ax, kwargs)
        else:
            self._PlotUtils._plot_individual_years(ax, kwargs.get('series_alpha', 0.3), kwargs)

        self._PlotUtils._customize_plot(ax, kwargs)
        self.fig = plt
        plt.show()

In [ ]:
SanJuan_df = PGPWData.getUSGSdata(start_date='2000-10-01', end_date=datetime.datetime.now().strftime('%Y-%m-%d')).reset_index()
a = DataLoader(SanJuan_df, 'dateTime', name_of_Q_column='value')
s = StatisticsCalculator(a)
StackedLinePlots(s,
                 title="SanJuan",
                 highlight_years=[2024],
                 spread_on=True,
                 spread_type='quartiles',
                 spread_alpha=0.5,
                 water_year_on=True,
#                  series_labels=True,
                 series_alpha=.15,
#                  y_upper_lim=110000, 
                 legend_pos="best",
                 legend_mode="partial",
                 legend_ncol=1,)

In [ ]:
# a = DataLoader(SanJuan_df, 'dateTime', name_of_Q_column='value')
# s = StatisticsCalculator(a)
# StackedLinePlots(s,
#                  title="SanJuan",
#                  highlight_years= None,
# #                  spread_on=True,
# #                  spread_type='quartiles',
# #                  spread_alpha=0.5,
#                  water_year_on='sfdg',
# #                  series_labels=True,
#                  series_alpha=.15,
# #                  y_upper_lim=110000, 
#                  legend_pos="best",
#                  legend_mode="partial",
#                  legend_ncol=1,)

In [ ]:
s._df['Calendar Year'].unique()

In [ ]:
s._grouped_water_years

In [ ]:
# s._df['Date'] = pd.to_datetime(s._df['Date'])
# s._df['WY_Date'] = s._df['Date']
# # s._df['Water Year'] = pd.to_datetime(s._df['Water Year'])

# for index, row in s._df.iterrows():
#     date = row['Date']
# #     print(row)
#     wy=row['Water Year']
# #     print(wy[index].year)
#     try:
#         new_date = date.replace(year=wy)
# #         print(f'Old Date: {date}')
# #         print(wy)
# #         print(new_date)

#     except ValueError:
#         new_date = date.replace(year=wy, day=28)
    
#     s._df.at[index, 'WY_Date']=new_date
    
# s._df

s_grouped = s._df.groupby('Water Year')

forced_x_positions=[1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 336],
forced_x_labels=['10-01', '11-01', '12-01', '01-01', '02-01', '03-01', '04-01', '05-01', '06-01', '07-01', '08-01', '09-01']
        

fig, ax = plt.subplots(figsize=(9, 7))

s_grouped.get_group(1922)

for i in s._df['Water Year'].unique():
    data = s_grouped.get_group(i)
    ax.plot(data['month-day'], data['value'])
    ax.set_xticks(forced_x_positions[0])
    ax.set_xticklabels(forced_x_labels, rotation=45)

In [ ]:
# s._pivot_table.plot(kind='line', stacked=True, figsize=(10,6))

In [ ]:
vip_sites = {

    #Upper Tribs
    'Okanogan River Near Tonasket, WA': 12445000,
    'Chelan River at Chelan': 12452500,
    #Yakima
    'Yakima at Kiona': 12510500,
    'Yakima River at Umtanum, WA': 12484500,
    #Pend Orielle
    'Clark Fork below Missoula MT': 12353000,
    'Clark Fork at St. Regis MT': 12354500,
    'Flathead at Columbia Falls MT': 12363000,
    #Spokane/Coeur D Alene
    'Spokane River near Post Falls, ID':  12419000,
    'Coeur D Alene River nr Cataldo, ID': 12413500,
    # Clearwater and Salmon
    'Clearwater River at Orofino ID': 13340000,      
    'MF Salmon River at Mouth Nr Shoup, ID': 13310199,
    #Snake River
        #Upper/headwaters
        #Mid/Boise
        #Lower
    'Snake River at Anatone': 13334300,
#     'Grande Ronde River at Troy, OR': 13333000,  #doesn't have 2024 data and causes script to crash...prob add error handling
#     'Snake River at Hells Canyon Dam': 13290450,
    'Snake River at Weiser ID': 13269000, 
    #Interior OR Basins
    'Owyhee River nr Rome OR': 13181000,
    'John Day at McDonald Ferry': 14048000,
    'John Day at Service Creek': 14046500,
    'Deschutes at Moody': 14103000,
    'Umatilla nr Pendleton': 14020850,
    #Cascades
    'Klickitat': 14113000,
    'Hood River': 14120000,
    'White Salmon': 14123500,
    'Walla Walla nr Touchet': 14018500,
}

In [ ]:
for site in vip_sites.items():
    print(site)
    data = PGPWData.getUSGSdata(start_date='2000-10-01', end_date=datetime.datetime.now().strftime('%Y-%m-%d'),sites=str(site[1])).reset_index()
    a = DataLoader(data, 'dateTime', name_of_Q_column='value')
    s = StatisticsCalculator(a)
    plot = StackedLinePlots(s,
                     water_year_on=True,
                     title=site[0],
                     highlight_years=[2024],
                     quartile_shading_on=True, 
                     quartile_shading_alpha=0.5,
                     series_labels=False,
                     series_alpha=.1,
                     y_upper_lim=20000,
                     y_lower_lim=0,
                     legend_pos="best",
                     legend_mode="partial",
                     legend_ncol=1,)
    
#     with open('historicalplots_matplotlib.html', 'a') as f:
#         f.write(plot.fig.to_html(full_html=False, include_plotlyjs='cdn'))


In [ ]:
Spokane_sites = {
    "CDA River near Cataldo": 12413500,
    "St. Joe River at Red Ives Ranger Station, ID": 12413875,
    "Spokane River nr Post Falls, ID": 12419000
}

In [ ]:
for site in Spokane_sites.items():
#     print(site)
    data = PGPWData.getUSGSdata(start_date='2000-10-01', end_date=datetime.datetime.now().strftime('%Y-%m-%d'),sites=str(site[1])).reset_index()
    a = DataLoader(data, 'dateTime', name_of_Q_column='value')
    s = StatisticsCalculator(a)
    StackedLinePlots(s,
                     water_year_on=True,
                     title=site[0],
                     highlight_years=[2024],
                     quartile_shading_on=True, 
                     quartile_shading_alpha=0.5,
                     series_labels=False,
                     series_alpha=.1,
#                      y_upper_lim=20000,
                     y_lower_lim=0,
                     legend_pos="best",
                     legend_mode="partial",
                     legend_ncol=1,)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import math
import calendar
# from scipy.integrate import trapz
from collections import OrderedDict

class StackedLinePlot:
    def __init__(self, csv_path, name_of_date_column, name_of_Q_column):
        self.csv_path = csv_path
        self._name_of_date_column = name_of_date_column
        self._name_of_Q_column = name_of_Q_column
        self.df = csv_path
        self._df = self.df
        self._df_stat_summary = self._df.describe()
        self._df['Date'] = pd.to_datetime(self._df[name_of_date_column])
        self._df = self._df[~self._df.duplicated('Date')]
        self._df['month'] = self._df['Date'].dt.month
        self._df['Year'] = self._df['Date'].dt.year
        self._df['month-day'] = self._df['Date'].apply(lambda x: x.strftime('%m-%d'))
        self._df['Water Year'] = self._df['Date'].dt.year.where(self._df['Date'].dt.month<10, self._df['Date'].dt.year+1)
        # self._pivot_table = self._df.pivot(index="month-day", columns='Year', values=name_of_Q_column)
        # self._pivot_table_monthly = self._df.pivot(columns='month', values=name_of_Q_column)
        # self._pivot_table_yearly_stats = {year: self._pivot_table.iloc[:, i].describe() for i, year in enumerate(self._pivot_table.columns)}
        self._forced_x_positions = None
        self._forced_x_labels = None
        self._mean = None
        self._median = None
        self._st_dev = None
        self._lower_bound = None
        self._upper_bound = None
        self._lower_bound_percentile25 = None
        self._upper_bound_percentile75 = None
        self._colors = ['crimson', 'springgreen', 'dodgerblue', 'purple', 'green', 'deeppink', "lawngreen", "coral", "lime", "navy", "goldenrod", 'crimson', 'springgreen', 'dodgerblue', 'purple', 'green', 'deeppink', "lawngreen", "coral", "lime", "navy", "goldenrod"]
        # self._ylim_max = self.df[name_of_Q_column].max() + self.df[name_of_Q_column].max() * 0.05
        # print(self._ylim_max)

    @property
    def df(self):
        return self._df

    @df.setter
    def df(self, csv_path):
        if isinstance(csv_path, pd.DataFrame):
            self._df = csv_path.copy()  # Set the DataFrame directly
            # Convert columns to numeric, handling errors by setting non-numeric values to NaN
            if isinstance(self._df[self._name_of_Q_column][0], str):
                self._df[self._name_of_Q_column] = pd.to_numeric(self._df[self._name_of_Q_column], errors='coerce')

        elif isinstance(csv_path, str):
            try:
                print(f"Importing CSV with the file path: {csv_path}")
                self._df = pd.read_csv(csv_path)
                # Convert columns to numeric, handling errors by setting non-numeric values to NaN
                if isinstance(self._df[self._name_of_Q_column][0], str):
                    self._df[self._name_of_Q_column] = pd.to_numeric(self._df[self._name_of_Q_column], errors='coerce')
                # self._df['Year'] = pd.to_numeric(self._df['Year'], errors='coerce')
            except Exception as e:
                print(f"Error reading CSV file: {e}")
                
        else:
            print("Invalid input type. Please provide either a pandas DataFrame or a CSV file path.")
            


    def calculate_statistics(self, **kwargs):

        self._stats = self._df.groupby("month-day")[self._name_of_Q_column].agg(['mean', 'median', 'std', ("q25", lambda x: x.quantile(0.25)), ("q75", lambda y: y.quantile(0.75))])
        self._monthly_stats = self._df.groupby("month")[self._name_of_Q_column].agg(['mean', 'median', 'std', ("q25", lambda x: x.quantile(0.25)), ("q75", lambda y: y.quantile(0.75))])
        self._mean = self._stats.iloc[:, 0]
        self._median = self._stats.iloc[:, 1]
        self._st_dev = self._stats.iloc[:, 2]
        self._percentile25 = self._stats.iloc[:, 3]
        self._percentile75 = self._stats.iloc[:, 4]
        self._lower_bound_st_dev = self._mean - self._st_dev
        self._upper_bound_st_dev = self._mean + self._st_dev
        self._lower_bound_percentile25 = self._mean - self._percentile25
        self._upper_bound_percentile75 = self._mean + self._percentile75

        water_year = kwargs.get('water_year', True)            
        year = 'Water Year' if water_year else 'Year'
        start_year = kwargs.get("start_year", self._df[year].iloc[0])
        end_year = kwargs.get("end_year", self._df[year].iloc[-1])
        self._df = self._df[(self._df[year] >= start_year) & (self._df[year] <= end_year)]
        self._unique_years = self._df[year].unique()
        self._start_year, self._end_year = self._unique_years[0], self._unique_years[-1]
        self._num_of_decades = math.ceil((self._end_year - self._start_year) / 10)
        self._unique_decades = self._df[year].apply(lambda year: (year // 10) * 10).unique()

        # Convert years in _pivot_table to water years
        water_year_col = self._df['Water Year'].unique()
        self._pivot_table = self._df.copy()
        self._pivot_table['month-day'] = self._pivot_table['Date'].apply(lambda x: x.strftime('%m-%d'))
        self._pivot_table = self._pivot_table.pivot(index="month-day", columns=year, values=self._name_of_Q_column)
        self._pivot_table = self._pivot_table[water_year_col]
        self._pivot_table_monthly = self._df.pivot(columns='month', values=self._name_of_Q_column)
        self._pivot_table_yearly_stats = {year: self._pivot_table.iloc[:, i].describe() for i, year in enumerate(self._pivot_table.columns)}

        # Call _prepare_data_for_plotting here
        # self._prepare_data_for_plotting(start_year, end_year, water_year=water_year)

    ##  KEEP!!!!!!  And fix up. - calculates volume under curve/water supply for the year
    # def calculate_yearly_volumes(self):
    #     years = []
    #     area = []
    #     for year in self._pivot_table.columns:
    #         dates = list(range(0, len(self._pivot_table[year].dropna())))
    #         area.append(trapz(self._pivot_table[year].dropna(), dates))
    #         years.append(year)

    #     Area_dict = OrderedDict()
    #     for key, value in zip(years, area):
    #         Area_dict[key] = value
        # return Area_dict


    def plotStackedLinePlot(self, **kwargs):
        '''
        Keyword Arguments and defaults include:
            plot_central_tendency_stats=True,
            highlight_years=[],
            water_year=True,
            quartile_shading=True,
            quartile_shading_alpha=0.5,
            group_by_decade=False,
            series_alpha=0.3,
            quartile_shading_zorder=1,
            'forced_x_positions'=[1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 336],
            'forced_x_labels'=['01-01', '02-01', '03-01', '04-01', '05-01', '06-01', '07-01', '08-01', '09-01', '10-01', '11-01', '12-01']
            'y_lower_lim'=0,
            'y_upper_lim'=25,
            'ylabel'='Discharge',
            'title',
            'legend_mode'='partial',
            'legend_pos'='upper right',
            'legend_ncol'=1
        '''

        # self._prepare_data_for_plotting(kwargs.get('input_start_year', 2010), kwargs.get('input_end_year', 2020))
        self.calculate_statistics()
        fig, ax = plt.subplots(figsize=(9, 7))

        self._plot_central_tendency_stats(ax, kwargs.get('plot_central_tendency_stats', True))
        self._plot_highlighted_years(ax, kwargs.get('highlight_years'))
        self._plot_quartile_shading(ax, kwargs.get('quartile_shading', True), kwargs.get('quartile_shading_alpha', 0.5), kwargs)

        if kwargs.get('group_by_decade', False):
            self._plot_grouped_by_decade(ax, kwargs)
        else:
            self._plot_individual_years(ax, kwargs.get('series_alpha', 0.3), kwargs)

        self._customize_plot(ax, kwargs)
        plt.show()


    def _plot_central_tendency_stats(self, ax, plot_stats):
        if plot_stats:
            self._mean.plot(ax=ax, label="Mean", linestyle=':', color='black', linewidth=1.5, zorder=3)
            self._median.plot(ax=ax, label="Median", linestyle=':', color='red', linewidth=1.5, zorder=3)

    def _plot_highlighted_years(self, ax, highlight_years):
        for i, year in enumerate(highlight_years):
            # if water_year:
            #     year = 'Water Year'
            # else:
            #     year = 'Year'

            self._pivot_table[year].plot(ax=ax, linewidth=1.6, zorder=3, color=self._colors[i])

    def _plot_quartile_shading(self, ax, quartile_shading, quartile_shading_alpha, kwargs):
       

        if quartile_shading:
            zorder = kwargs.get('quartile_shading_zorder', 1)

            plt.fill_between(
                list(range(0, len(pd.DataFrame(self._mean).iloc[:, 0]))),
                pd.DataFrame(self._mean).iloc[:, 0].astype(float),
                pd.DataFrame(self._lower_bound_st_dev).iloc[:, 0].astype(float),
                where=(pd.DataFrame(self._mean).iloc[:, 0].astype(float) > pd.DataFrame(self._lower_bound_percentile25).iloc[:, 0].astype(float)),
                interpolate=True, color='yellow', alpha=quartile_shading_alpha, zorder=zorder, label="q25-q75")

            plt.fill_between(
                list(range(0, len(pd.DataFrame(self._mean).iloc[:, 0]))),
                pd.DataFrame(self._mean).iloc[:, 0].astype(float),
                pd.DataFrame(self._upper_bound_st_dev).iloc[:, 0].astype(float),
                where=(pd.DataFrame(self._mean).iloc[:, 0].astype(float) < pd.DataFrame(self._upper_bound_percentile75).iloc[:, 0].astype(float)),
                interpolate=True, color='yellow', alpha=quartile_shading_alpha, zorder=zorder)

    def _plot_grouped_by_decade(self, ax, kwargs):
        for i, decade in enumerate(self._unique_decades):
            years_in_decade = [year for year in self._unique_years if decade <= year < decade + 10]
            mean_values = self._pivot_table[years_in_decade].mean(axis=1)
            std_dev_values = self._pivot_table[years_in_decade].std(axis=1)
            ax.plot(self._pivot_table.index, mean_values, label=f'Decade {decade}s', color=self._colors[i])
            ax.fill_between(self._pivot_table.index, mean_values - std_dev_values, mean_values + std_dev_values, alpha=0.2, color=self._colors[i])

    def _plot_individual_years(self, ax, series_alpha, kwargs):
        for i, year in enumerate(self._unique_years):
            ax.plot(self._pivot_table.index, self._pivot_table[year], label=f'{year}', alpha=series_alpha)

    def _customize_plot(self, ax, kwargs):
        self._forced_x_positions = kwargs.get('forced_x_positions', [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 336]),
        self._forced_x_labels = kwargs.get('forced_x_labels', ['01-01', '02-01', '03-01', '04-01', '05-01', '06-01', '07-01', '08-01', '09-01', '10-01', '11-01', '12-01'])

        if self._forced_x_positions is not None and self._forced_x_labels is not None:
            print(self._forced_x_positions[0])
            print(self._forced_x_labels)
            ax.set_xticks(self._forced_x_positions[0])
            ax.set_xticklabels(self._forced_x_labels, rotation=45)
            xlim_min = self._forced_x_positions[0][0]
            xlim_max = self._forced_x_positions[0][-1]
            ax.set_xlim([xlim_min, xlim_max])
            ax.set_ylim([kwargs.get('y_lower_lim', self._df[self._name_of_Q_column].min()), kwargs.get('y_upper_lim', self._df[self._name_of_Q_column].max())])

        plt.grid(color='green', linestyle=":", linewidth=0.5)
        plt.xlabel('Month-Day')
        plt.ylabel(kwargs.get('ylabel', "Discharge (cfs)"))
        plt.title(kwargs.get('title'))

        if kwargs.get('legend_mode') == "partial":
#             labels = ['Mean', 'Median', 'q25-q75'] + kwargs.get('highlight_years', [])
            plt.legend(loc=kwargs.get('legend_pos', 'upper right'), ncol=kwargs.get('legend_ncol', 1), labels=['Mean', 'Median', 'q25-q75'] + kwargs.get('highlight_years', []))
        else:
            plt.legend(loc=kwargs.get('legend_pos'), ncol=kwargs.get('legend_ncol'))#, labels=labels)

if __name__ == "__main__":
    print("-------This module creates customized StackedLinePlots.-------")

In [ ]:
# class StatisticsCalculator:
#     def __init__(self, DataLoader):
#         self.data_loader = DataLoader
#         self._df = self.data_loader._df
#         self._df['Date'] = pd.to_datetime(self._df[self.data_loader._name_of_date_column])
#         self._df = self._df[~self._df.duplicated('Date')]
#         self._df['month'] = self._df['Date'].dt.month
#         self._df['Year'] = self._df['Date'].dt.year
#         self._df['month-day'] = self._df['Date'].apply(lambda x: x.strftime('%m-%d'))
#         self._df['Water Year'] = self._df['Date'].dt.year.where(self._df['Date'].dt.month<10, self._df['Date'].dt.year+1)
#         self.stats = self.calculate_statistics()
    
#     def calculate_statistics(self, **kwargs):

#         self._stats = self._df.groupby("month-day")[self.data_loader._name_of_Q_column].agg(['mean', 'median', 'std', ("q25", lambda x: x.quantile(0.25)), ("q75", lambda y: y.quantile(0.75))])
#         self._monthly_stats = self._df.groupby("month")[self.data_loader._name_of_Q_column].agg(['mean', 'median', 'std', ("q25", lambda x: x.quantile(0.25)), ("q75", lambda y: y.quantile(0.75))])
#         self._mean = self._stats.iloc[:, 0]
#         self._median = self._stats.iloc[:, 1]
#         self._st_dev = self._stats.iloc[:, 2]
#         self._percentile25 = self._stats.iloc[:, 3]
#         self._percentile75 = self._stats.iloc[:, 4]
#         self._lower_bound_st_dev = self._mean - self._st_dev
#         self._upper_bound_st_dev = self._mean + self._st_dev
#         self._lower_bound_percentile25 = self._mean - self._percentile25
#         self._upper_bound_percentile75 = self._mean + self._percentile75

#         water_year = kwargs.get('water_year', True)            
#         year = 'Water Year' if water_year else 'Year'
#         start_year = kwargs.get("start_year", self._df[year].iloc[0])
#         end_year = kwargs.get("end_year", self._df[year].iloc[-1])
#         self._df = self._df[(self._df[year] >= start_year) & (self._df[year] <= end_year)]
#         self._unique_years = self._df[year].unique()
#         self._start_year, self._end_year = self._unique_years[0], self._unique_years[-1]
#         self._num_of_decades = math.ceil((self._end_year - self._start_year) / 10)
#         self._unique_decades = self._df[year].apply(lambda year: (year // 10) * 10).unique()

#         # Convert years in _pivot_table to water years
#         water_year_col = self._df['Water Year'].unique()
#         self._pivot_table = self._df.copy()
#         self._pivot_table['month-day'] = self._pivot_table['Date'].apply(lambda x: x.strftime('%m-%d'))
#         self._pivot_table = self._pivot_table.pivot(index="month-day", columns=year, values=self.data_loader._name_of_Q_column)
#         self._pivot_table = self._pivot_table[water_year_col]
#         self._pivot_table_monthly = self._df.pivot(columns='month', values=self.data_loader._name_of_Q_column)
#         self._pivot_table_yearly_stats = {year: self._pivot_table.iloc[:, i].describe() for i, year in enumerate(self._pivot_table.columns)}

#         # Call _prepare_data_for_plotting here
#         # self._prepare_data_for_plotting(start_year, end_year, water_year=water_year)

#     ##  KEEP!!!!!!  And fix up. - calculates volume under curve/water supply for the year
#     # def calculate_yearly_volumes(self):
#     #     years = []
#     #     area = []
#     #     for year in self._pivot_table.columns:
#     #         dates = list(range(0, len(self._pivot_table[year].dropna())))
#     #         area.append(trapz(self._pivot_table[year].dropna(), dates))
#     #         years.append(year)

#     #     Area_dict = OrderedDict()
#     #     for key, value in zip(years, area):
#     #         Area_dict[key] = value
#         # return Area_dict

    

In [ ]:
# class PlotUtils:
#     def __init__(self, StatisticsCalculator):
#         self._StatisticsCalculator = StatisticsCalculator
#         self._colors = ['crimson', 'springgreen', 'dodgerblue', 'purple', 'green', 'deeppink', "lawngreen", "coral", "lime", "navy", "goldenrod", 'crimson', 'springgreen', 'dodgerblue', 'purple', 'green', 'deeppink', "lawngreen", "coral", "lime", "navy", "goldenrod"]


#     def _plot_central_tendency_stats(self, ax, plot_stats):
#         if plot_stats:
#             self._StatisticsCalculator._mean.plot(ax=ax, label="Mean", linestyle=':', color='black', linewidth=1.5, zorder=3)
#             self._StatisticsCalculator._median.plot(ax=ax, label="Median", linestyle=':', color='red', linewidth=1.5, zorder=3)

#     def _plot_highlighted_years(self, ax, highlight_years):
#         for i, year in enumerate(highlight_years):
#             # if water_year:
#             #     year = 'Water Year'
#             # else:
#             #     year = 'Year'

#             self._StatisticsCalculator._pivot_table[year].plot(ax=ax, linewidth=1.6, zorder=3, color=self._colors[i])

#     def _plot_quartile_shading(self, ax, quartile_shading, quartile_shading_alpha, kwargs):
       

#         if quartile_shading:
#             zorder = kwargs.get('quartile_shading_zorder', 1)

#             plt.fill_between(
#                 list(range(0, len(pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0]))),
#                 pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0].astype(float),
#                 pd.DataFrame(self._StatisticsCalculator._lower_bound_st_dev).iloc[:, 0].astype(float),
#                 where=(pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0].astype(float) > pd.DataFrame(self._StatisticsCalculator._lower_bound_percentile25).iloc[:, 0].astype(float)),
#                 interpolate=True, color='yellow', alpha=quartile_shading_alpha, zorder=zorder, label="q25-q75")

#             plt.fill_between(
#                 list(range(0, len(pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0]))),
#                 pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0].astype(float),
#                 pd.DataFrame(self._StatisticsCalculator._upper_bound_st_dev).iloc[:, 0].astype(float),
#                 where=(pd.DataFrame(self._StatisticsCalculator._mean).iloc[:, 0].astype(float) < pd.DataFrame(self._StatisticsCalculator._upper_bound_percentile75).iloc[:, 0].astype(float)),
#                 interpolate=True, color='yellow', alpha=quartile_shading_alpha, zorder=zorder)

#     def _plot_grouped_by_decade(self, ax, kwargs):
#         for i, decade in enumerate(self._StatisticsCalculator._unique_decades):
#             years_in_decade = [year for year in self._StatisticsCalculator._unique_years if decade <= year < decade + 10]
#             mean_values = self._StatisticsCalculator._pivot_table[years_in_decade].mean(axis=1)
#             std_dev_values = self._StatisticsCalculator._pivot_table[years_in_decade].std(axis=1)
#             ax.plot(self._StatisticsCalculator._pivot_table.index, mean_values, label=f'Decade {decade}s', color=self._colors[i])
#             ax.fill_between(self._StatisticsCalculator._pivot_table.index, mean_values - std_dev_values, mean_values + std_dev_values, alpha=0.2, color=self._colors[i])

#     def _plot_individual_years(self, ax, series_alpha, kwargs):
#         for i, year in enumerate(self._StatisticsCalculator._unique_years):
#             ax.plot(self._StatisticsCalculator._pivot_table.index, self._StatisticsCalculator._pivot_table[year], label=f'{year}', alpha=series_alpha)

#     def _customize_plot(self, ax, kwargs):
#         self._forced_x_positions = kwargs.get('forced_x_positions', [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 336]),
#         self._forced_x_labels = kwargs.get('forced_x_labels', ['01-01', '02-01', '03-01', '04-01', '05-01', '06-01', '07-01', '08-01', '09-01', '10-01', '11-01', '12-01'])

#         if self._forced_x_positions is not None and self._forced_x_labels is not None:
#             print(self._forced_x_positions[0])
#             print(self._forced_x_labels)
#             ax.set_xticks(self._forced_x_positions[0])
#             ax.set_xticklabels(self._forced_x_labels, rotation=45)
#             xlim_min = self._forced_x_positions[0][0]
#             xlim_max = self._forced_x_positions[0][-1]
#             ax.set_xlim([xlim_min, xlim_max])
#             ax.set_ylim([kwargs.get('y_lower_lim', self._StatisticsCalculator._df[self._StatisticsCalculator.data_loader._name_of_Q_column].min()), kwargs.get('y_upper_lim', self._StatisticsCalculator._df[self._StatisticsCalculator.data_loader._name_of_Q_column].max())])

#         plt.grid(color='green', linestyle=":", linewidth=0.5)
#         plt.xlabel('Month-Day')
#         plt.ylabel(kwargs.get('ylabel', "Discharge (cfs)"))
#         plt.title(kwargs.get('title'))

#         if kwargs.get('legend_mode') == "partial":
# #             labels = ['Mean', 'Median', 'q25-q75'] + kwargs.get('highlight_years', [])
#             plt.legend(loc=kwargs.get('legend_pos', 'upper right'), ncol=kwargs.get('legend_ncol', 1), labels=['Mean', 'Median', 'q25-q75'] + kwargs.get('highlight_years', []))
#         else:
#             plt.legend(loc=kwargs.get('legend_pos'), ncol=kwargs.get('legend_ncol'))#, labels=labels)
